# Stage 1: features + random forest + HMM (the baseline)

The CNN in Stage 2 has to beat this. Steps:

1. Split by person
2. Turn each 30 s window into 16 simple features
3. Choose one forest setting using training people only
4. Train the forest, then smooth its answers with the HMM
5. Score on the test people, before and after the HMM

The split, HMM and metrics live in `common.py`, so Stage 2 uses exactly the same ones.

**Kernel:** `anaconda3`. Run `deeplearn.ipynb` (Stage 0) first.

## How this works: random forest + HMM, explained from scratch

**The task, in one line:** take 30 seconds of wrist movement (1500 readings x 3 directions) and say which of 4 activities it was: *sleep, sedentary, light, moderate-vigorous*.

### Step 1: describe each window with 16 numbers (features)

A tree model can't make sense of 4500 raw numbers, so we describe each window the way a person would:

| Question a person would ask | Feature(s) |
|---|---|
| How much is the wrist shaking? | standard deviation of the magnitude and of x, y, z |
| How strong are the biggest jolts? | max, min, range of the magnitude |
| Which way is the wrist pointing? | mean of x, y, z, plus pitch and roll (gravity tells us the angle) |
| Is there a rhythm, like steps? | dominant frequency, power at 0.5–3 Hz (walking) and 3–8 Hz (faster) |

*Example:* sleeping gives tiny shaking and a wrist lying still at one angle. Walking gives clear shaking with a beat of about 2 steps a second.

### Step 2: one decision tree = a flowchart learned from data

A decision tree asks yes/no questions about the features, one after another. A made-up small example (the real trees are much bigger):

```
is the shaking (mag_std) < 0.01 g?
├── yes: is the wrist tilted like lying in bed (roll)?
│        ├── yes -> sleep
│        └── no  -> sedentary
└── no:  is there a strong 1–3 Hz rhythm?
         ├── yes -> moderate-vigorous
         └── no  -> light
```

Nobody writes these questions. Training tries many possible questions and keeps the one that best separates the labels, then repeats inside each branch.

### Step 3: a forest = 200 different trees that vote

One tree tends to **memorise** its training data, including its accidents. So we grow 200 trees, each made a little different:

- each tree trains on a random sample of the windows
- at each question, each tree may only choose from a random few of the 16 features

Every tree makes different mistakes. When they **vote**, the mistakes mostly cancel out. The share of votes becomes the probability, e.g. 140 of 200 trees say "sleep" = 70% sleep.

### Step 4: decisions we made, and why

| Decision | Why |
|---|---|
| Test people are kept completely apart (split **by person**) | Windows from one person look alike. Testing on someone the model trained on would make it look better than it will be on a new watch wearer |
| `class_weight='balanced'` | Moderate-vigorous is only about 5% of the data. Without weights the forest could almost ignore it and still score well |
| The leaf size (`min_samples_leaf`) is chosen by 5-fold validation **grouped by person**, on training people only | Settings must never be chosen by peeking at the test people |

### The HMM: fixing one window at a time with common sense

The model judges each 30 s window **on its own**. But people don't switch activity every 30 seconds. If the model says

`sleep, sleep, light, sleep, sleep`

at 3 am, that lone "light" is almost certainly a mistake (maybe the person rolled over).

The **HMM** (hidden Markov model) fixes this by looking at the whole day at once. It weighs two things:

1. **What the model says** about each window (its probabilities, e.g. "70% sleep, 25% light")
2. **How often people really switch** from one activity to another, counted from the training people's labels (in the training people, the next window after sleep is sleep again 99.9% of the time; after light it is light 95% of the time)

The **Viterbi** algorithm then finds the single most likely sequence of labels for the whole day. The lone "light" gets overruled, and you get `sleep, sleep, sleep, sleep, sleep`. A long run of "light" in the morning would be kept, because the evidence is strong for many windows in a row.

### How the results are scored

| Score | What it asks | Why we use it |
|---|---|---|
| **Accuracy** | What share of windows is right? | Easy to read, but flattering: always guessing "sedentary" already gets about 40% |
| **Macro-F1** | How good is it on *each* class, averaged with equal weight? | The rare moderate-vigorous class counts as much as sleep |
| **Cohen's kappa** | How much better than lucky guessing? 0 = guessing, 1 = perfect | The fairest single number when classes are uneven |

Each is reported for the **4 classes** and for **binary** (still = sleep + sedentary, moving = light + moderate-vigorous). Binary is the simple "is this person moving?" answer the watch needs most.

In [1]:
import itertools
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

from common import (BINARY, CLASSES, DATA_DIR, OLD_TEST_FILE, RESULTS_DIR, SEED, confusion_table, evaluate, learn_hmm,
                    scores, smooth, split_people, to_binary, viterbi)

RESULTS_DIR.mkdir(exist_ok=True)

## 1. Split by person

Test = the 11 people from the earlier work + 19 random others (fixed seed). The other 121 are for training. The test people are not used for any choice below.

In [2]:
train_ids, test_ids = split_people()
assert not set(train_ids) & set(test_ids) and len(train_ids) == 121 and len(test_ids) == 30

old_test = set(pd.read_csv(OLD_TEST_FILE)['participant_id'])
assert old_test <= set(test_ids)
print('old 11  :', sorted(old_test))
print('new 19  :', sorted(set(test_ids) - old_test))

ages = pd.read_csv('../../data/open datasets/capture24/capture24/metadata.csv').set_index('pid')['age']
print('\npeople per age group:')
print(pd.DataFrame({'train': ages[train_ids].value_counts(), 'test': ages[test_ids].value_counts()}).sort_index().to_string())
pd.DataFrame({'participant_id': test_ids, 'from_old_test': [p in old_test for p in test_ids]}).to_csv(
    RESULTS_DIR / 'test_participants.csv', index=False)

old 11  : ['P012', 'P057', 'P064', 'P076', 'P081', 'P092', 'P096', 'P107', 'P111', 'P117', 'P137']
new 19  : ['P011', 'P013', 'P014', 'P019', 'P028', 'P056', 'P073', 'P074', 'P087', 'P097', 'P103', 'P105', 'P106', 'P112', 'P118', 'P120', 'P128', 'P136', 'P141']

people per age group:
       train  test
age               
18-29     34     9
30-37     35     2
38-52     27    10
53+       25     9


## 2. Features

| Group | Features | What they tell the forest |
|---|---|---|
| Strength | magnitude mean, std, min, max, range | how much the wrist moves |
| Axes | mean and std of x, y, z | direction of the wrist and which way it moves |
| Wrist angle | pitch, roll | how the wrist is tilted, from the direction of gravity |
| Rhythm | dominant frequency, power in 0.5-3 Hz and 3-8 Hz | a regular beat such as walking (about 2 Hz), or faster hand movement |

The dominant frequency is searched inside 0.5-8 Hz. Below 0.5 Hz is slow drift (such as turning over), not rhythm.

In [3]:
HZ, WIN = 50, 1500
FREQS = np.fft.rfftfreq(WIN, 1 / HZ)               # the frequency of each FFT step (0.033 Hz apart)
LOW_BAND = (FREQS >= 0.5) & (FREQS < 3)
HIGH_BAND = (FREQS >= 3) & (FREQS <= 8)
BOTH_BANDS = LOW_BAND | HIGH_BAND

FEATURES = ['mag_mean', 'mag_std', 'mag_min', 'mag_max', 'mag_range',
            'x_mean', 'y_mean', 'z_mean', 'x_std', 'y_std', 'z_std',
            'pitch', 'roll', 'dom_freq', 'power_0.5_3', 'power_3_8']


def make_features(X):
    """(n, 1500, 3) windows in g -> (n, 16) features."""
    X = X.astype('float64')                        # WHY: more exact sums over 1500 small numbers
    mag = np.sqrt((X ** 2).sum(axis=2))
    mean, std = X.mean(axis=1), X.std(axis=1)

    # WHY the mean: over 30 s, movement mostly averages out and what is left points along gravity
    pitch = np.degrees(np.arctan2(-mean[:, 0], np.sqrt(mean[:, 1] ** 2 + mean[:, 2] ** 2)))
    roll = np.degrees(np.arctan2(mean[:, 1], mean[:, 2]))

    # WHY subtract the mean: gravity is a constant 1 g and would swamp everything else in the FFT
    spectrum = np.abs(np.fft.rfft(mag - mag.mean(axis=1, keepdims=True), axis=1)) ** 2
    # WHY this scale: a band's power is then the variance that band adds to the magnitude (in g squared)
    spectrum = spectrum * 2 / WIN ** 2
    dom_freq = FREQS[BOTH_BANDS][spectrum[:, BOTH_BANDS].argmax(axis=1)]

    return np.column_stack([mag.mean(axis=1), mag.std(axis=1), mag.min(axis=1), mag.max(axis=1),
                            mag.max(axis=1) - mag.min(axis=1), mean, std, pitch, roll, dom_freq,
                            spectrum[:, LOW_BAND].sum(axis=1), spectrum[:, HIGH_BAND].sum(axis=1)])


# check on fake windows with known answers
t = np.arange(WIN) / HZ
still = np.zeros((1, WIN, 3)); still[..., 2] = 1                       # flat on a table
f = dict(zip(FEATURES, make_features(still)[0]))
assert np.isclose(f['mag_mean'], 1) and np.isclose(f['mag_std'], 0) and np.isclose(f['pitch'], 0) and np.isclose(f['roll'], 0)

walk = still.copy(); walk[0, :, 2] += 0.3 * np.sin(2 * np.pi * 2 * t)  # 2 Hz beat
f = dict(zip(FEATURES, make_features(walk)[0]))
assert np.isclose(f['dom_freq'], 2) and np.isclose(f['power_0.5_3'], 0.3 ** 2 / 2, rtol=0.01) and f['power_3_8'] < 1e-6

side = np.zeros((1, WIN, 3)); side[..., 1] = 1                         # rolled 90 degrees
assert np.isclose(dict(zip(FEATURES, make_features(side)[0]))['roll'], 90)
down = np.zeros((1, WIN, 3)); down[..., 0] = -1                        # pitched 90 degrees
assert np.isclose(dict(zip(FEATURES, make_features(down)[0]))['pitch'], 90)
print('feature checks passed')

feature checks passed


In [4]:
def load_features(ids):
    """One person at a time -> features, labels, person id, start time (in time order within each person)."""
    F, y, pid, start = [], [], [], []
    for p in ids:
        with np.load(DATA_DIR / f'{p}.npz') as f:
            F.append(make_features(f['X'])); y.append(f['y4']); start.append(f['start'])
        pid.append(np.full(len(y[-1]), p))
    return np.concatenate(F), np.concatenate(y).astype(int), np.concatenate(pid), np.concatenate(start)

t0 = time.time()
F_train, y_train, pid_train, _ = load_features(train_ids)
F_test, y_test, pid_test, start_test = load_features(test_ids)
assert np.isfinite(F_train).all() and np.isfinite(F_test).all()
assert all((np.diff(start_test[pid_test == p]) > 0).all() for p in test_ids)      # time order, needed by the HMM
print(f'train {F_train.shape}, test {F_test.shape}, {time.time() - t0:.0f} s')

share = lambda y: (100 * np.bincount(y, minlength=4) / len(y)).round(1)
print(pd.DataFrame({'train %': share(y_train), 'test %': share(y_test), 'test windows': np.bincount(y_test, minlength=4)},
                   index=CLASSES).to_string())

train (243012, 16), test (58152, 16), 42 s
                   train %  test %  test windows
sleep                 37.2    39.5         22947
sedentary             39.2    41.4         24075
light                 18.8    14.5          8439
moderate-vigorous      4.8     4.6          2691


## 3. Choose one setting, using training people only

`min_samples_leaf` is how many windows must end up in each leaf. Too small and the forest memorises people, too big and it blurs. Five-fold validation **grouped by person**: each fold is tested on people that fold never trained on. The setting with the best 4-class macro-F1 wins.

In [5]:
def make_forest(leaf):
    # WHY balanced: moderate-vigorous is rare, and without it the forest would mostly ignore that class
    return RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, class_weight='balanced',
                                  random_state=SEED, n_jobs=-1)

rows = []
for leaf in [1, 5, 20, 50]:
    fold_f1 = []
    for fit_idx, val_idx in GroupKFold(5).split(F_train, y_train, pid_train):
        assert not set(pid_train[fit_idx]) & set(pid_train[val_idx])
        forest = make_forest(leaf).fit(F_train[fit_idx], y_train[fit_idx])
        fold_f1.append(scores(y_train[val_idx], forest.predict(F_train[val_idx]), 4)['macro-F1'])
    rows.append({'min_samples_leaf': leaf, 'mean macro-F1': np.mean(fold_f1), 'std over folds': np.std(fold_f1)})
tuning = pd.DataFrame(rows)
best_leaf = int(tuning.loc[tuning['mean macro-F1'].idxmax(), 'min_samples_leaf'])
tuning.to_csv(RESULTS_DIR / 'stage1_tuning.csv', index=False)
print(tuning.round(3).to_string(index=False))
print('chosen:', best_leaf)

 min_samples_leaf  mean macro-F1  std over folds
                1          0.716           0.009
                5          0.748           0.016
               20          0.743           0.019
               50          0.733           0.021
chosen: 5


## 4. HMM: check it, then learn it

Viterbi is checked against the slow but obvious way: try **every** possible sequence and keep the best.

In [6]:
# 1. against brute force on small random problems
rng = np.random.default_rng(0)
for _ in range(50):
    probs = rng.dirichlet(np.ones(4), size=5)
    trans = rng.dirichlet(np.ones(4), size=4)
    first = rng.dirichlet(np.ones(4))
    def path_score(path):
        s = np.log(first[path[0]]) + np.log(probs[0, path[0]])
        for i in range(1, 5):
            s += np.log(trans[path[i - 1], path[i]]) + np.log(probs[i, path[i]])
        return s
    best = max(itertools.product(range(4), repeat=5), key=path_score)
    assert tuple(viterbi(probs, first, trans)) == best

# 2. what we want it to do: fix one odd window, keep a long real change
sticky = np.full((4, 4), 0.01) + np.eye(4) * 0.96
def confident(labels):
    return np.full((len(labels), 4), 0.1) + np.eye(4)[labels] * 0.6
flat = np.ones(4) / 4
assert list(viterbi(confident([1] * 10 + [2] + [1] * 10), flat, sticky)) == [1] * 21
assert list(viterbi(confident([1] * 10 + [2] * 10), flat, sticky)) == [1] * 10 + [2] * 10

# 3. a step from one person to the next is not counted
_, T = learn_hmm(np.array([0, 0, 1, 1]), np.array(['a', 'a', 'b', 'b']))
assert np.isclose(T[0, 1], 1 / 5)                  # only the "+1" is there, not a counted 0 -> 1 step
print('HMM checks passed')

start_p, transition = learn_hmm(y_train, pid_train)
assert np.allclose(transition.sum(axis=1), 1)
print('\nlearned transition matrix (row = now, column = next window):')
print(pd.DataFrame(transition, index=CLASSES, columns=CLASSES).round(4).to_string())
np.savez(RESULTS_DIR / 'hmm.npz', start=start_p, transition=transition)

HMM checks passed

learned transition matrix (row = now, column = next window):
                    sleep  sedentary   light  moderate-vigorous
sleep              0.9987     0.0003  0.0010             0.0001
sedentary          0.0006     0.9797  0.0189             0.0008
light              0.0011     0.0398  0.9483             0.0107
moderate-vigorous  0.0006     0.0076  0.0406             0.9512


## 5. Train on all training people and score on the test people

The HMM uses the forest's probabilities as its emissions. The binary answer is the 4-class answer merged into still/moving, so there is one model and not two.

In [7]:
t0 = time.time()
forest = make_forest(best_leaf).fit(F_train, y_train)
print(f'trained in {time.time() - t0:.0f} s')

assert list(forest.classes_) == [0, 1, 2, 3]       # probability columns are in our class order
probs = forest.predict_proba(F_test)
pred_rf = probs.argmax(axis=1)
pred_hmm = smooth(probs, pid_test, start_p, transition)

# reference: always answering the most common training class. A useful model must beat this clearly.
always = np.full(len(y_test), np.bincount(y_train).argmax())
table = pd.DataFrame(evaluate('always ' + CLASSES[always[0]], y_test, always)
                     + evaluate('RF', y_test, pred_rf) + evaluate('RF+HMM', y_test, pred_hmm))
table.to_csv(RESULTS_DIR / 'stage1_metrics.csv', index=False)
print(table.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa'])
      .loc[['always ' + CLASSES[always[0]], 'RF', 'RF+HMM']].round(3).to_string())

trained in 52 s
                 accuracy        macro-F1          kappa       
task              4-class binary  4-class binary 4-class binary
model                                                          
always sedentary    0.414  0.809    0.146  0.447   0.000  0.000
RF                  0.806  0.907    0.741  0.852   0.701  0.705
RF+HMM              0.880  0.933    0.801  0.889   0.814  0.778


In [8]:
for name, pred in [('RF', pred_rf), ('RF+HMM', pred_hmm)]:
    print(f'===== {name}: 4-class =====')
    print(confusion_table(y_test, pred, CLASSES).to_string())
    print(f'\n===== {name}: binary =====')
    print(confusion_table(to_binary(y_test), to_binary(pred), BINARY).to_string(), '\n')

===== RF: 4-class =====
                        sleep  sedentary  light  moderate-vigorous  sleep %  sedentary %  light %  moderate-vigorous %
true sleep              19681       2923    336                  7     85.8         12.7      1.5                  0.0
true sedentary           1460      20041   2492                 82      6.1         83.2     10.4                  0.3
true light                 49       2259   5559                572      0.6         26.8     65.9                  6.8
true moderate-vigorous      3        163    958               1567      0.1          6.1     35.6                 58.2

===== RF: binary =====
             still  moving  still %  moving %
true still   44105    2917     93.8       6.2
true moving   2474    8656     22.2      77.8 

===== RF+HMM: 4-class =====
                        sleep  sedentary  light  moderate-vigorous  sleep %  sedentary %  light %  moderate-vigorous %
true sleep              21664       1118    165                  0    

## 6. Is the gain real, or a few people?

Windows from one person are alike, so the honest sample size is 30 people and not thousands of windows. Score each person separately.

In [9]:
people = []
for p in test_ids:
    m = pid_test == p
    a, b = scores(y_test[m], pred_rf[m], 4), scores(y_test[m], pred_hmm[m], 4)
    a2, b2 = [scores(to_binary(y_test[m]), to_binary(q[m]), 2) for q in (pred_rf, pred_hmm)]
    people.append({'pid': p, 'age': ages[p], 'old_test': p in old_test, 'rf_kappa': a['kappa'], 'hmm_kappa': b['kappa'],
                   'rf_kappa_bin': a2['kappa'], 'hmm_kappa_bin': b2['kappa'],
                   'changes_true': int((np.diff(y_test[m]) != 0).sum()), 'changes_rf': int((np.diff(pred_rf[m]) != 0).sum()),
                   'changes_hmm': int((np.diff(pred_hmm[m]) != 0).sum())})
people = pd.DataFrame(people)
people.to_csv(RESULTS_DIR / 'stage1_per_person.csv', index=False)

print('kappa per person:')
print(people[['rf_kappa', 'hmm_kappa', 'rf_kappa_bin', 'hmm_kappa_bin']].describe().loc[['min', '25%', '50%', '75%', 'max']].round(3).to_string())
print('people the HMM improved: 4-class', int((people.hmm_kappa > people.rf_kappa).sum()), '/ 30, binary',
      int((people.hmm_kappa_bin > people.rf_kappa_bin).sum()), '/ 30')
print('\nlabel changes per person (median): true', people.changes_true.median(), ', RF', people.changes_rf.median(),
      ', RF+HMM', people.changes_hmm.median())
for label, mask in [('old 11', people.old_test), ('new 19', ~people.old_test), ('53+', people.age == '53+'), ('under 53', people.age != '53+')]:
    print(f'median kappa, {label:9s} (n={mask.sum():2d}): RF {people.rf_kappa[mask].median():.3f}, RF+HMM {people.hmm_kappa[mask].median():.3f}')

print('\nfeature importance (how much each feature is used; similar features share it):')
print(pd.Series(forest.feature_importances_, FEATURES).sort_values(ascending=False).round(3).to_string())

kappa per person:
     rf_kappa  hmm_kappa  rf_kappa_bin  hmm_kappa_bin
min     0.345      0.482         0.145          0.165
25%     0.623      0.771         0.586          0.640
50%     0.727      0.819         0.706          0.798
75%     0.765      0.873         0.784          0.861
max     0.874      0.960         0.917          0.958
people the HMM improved: 4-class 30 / 30, binary 28 / 30

label changes per person (median): true 33.5 , RF 331.5 , RF+HMM 26.5
median kappa, old 11    (n=11): RF 0.661, RF+HMM 0.799
median kappa, new 19    (n=19): RF 0.729, RF+HMM 0.853
median kappa, 53+       (n= 9): RF 0.661, RF+HMM 0.790
median kappa, under 53  (n=21): RF 0.729, RF+HMM 0.853

feature importance (how much each feature is used; similar features share it):
z_std          0.139
mag_std        0.130
x_std          0.101
y_std          0.098
power_0.5_3    0.076
power_3_8      0.072
mag_range      0.071
mag_mean       0.054
y_mean         0.039
mag_max        0.037
z_mean         0.037

## 7. Save

Saved for Stage 3: the model, the test probabilities and predictions, so the CNN can be compared person by person without retraining the forest.

In [10]:
joblib.dump({'model': forest, 'features': FEATURES, 'classes': CLASSES}, RESULTS_DIR / 'rf_4class.joblib', compress=3)
np.savez(RESULTS_DIR / 'stage1_predictions.npz', pid=pid_test, start=start_test, y=y_test,
         probs=probs, pred_rf=pred_rf, pred_hmm=pred_hmm)

# check the saved model gives the same answers as the one in memory
again = joblib.load(RESULTS_DIR / 'rf_4class.joblib')['model'].predict_proba(F_test[:2000])
assert np.allclose(again, probs[:2000])
for f in sorted(RESULTS_DIR.iterdir()):
    print(f'{f.name:28s} {f.stat().st_size / 1e6:8.2f} MB')

hmm.npz                          0.00 MB
rf_4class.joblib               136.15 MB
stage1_metrics.csv               0.00 MB
stage1_per_person.csv            0.00 MB
stage1_predictions.npz           4.19 MB
stage1_tuning.csv                0.00 MB
test_participants.csv            0.00 MB
